# Feature Build Examples

이 노트북은 `fb_*.py` feature build 모듈의 활용 예시입니다.

핵심 원칙:

- Stage가 아니라 `FeatureSpec` 목록이 생성 feature를 결정합니다.
- 초보자는 `COLUMN_MAP`과 `FEATURE_SPECS` 셀만 보고 입력 컬럼, 생성 컬럼, 파라미터를 검토할 수 있어야 합니다.
- full data 실행은 기본값이 아닙니다. 먼저 sample build로 catalog와 feature info를 확인합니다.
- 기존 산출물 overwrite는 기본 차단됩니다.


## 1. 경로와 실행 옵션

기본값은 제공하지만, 환경/경로 의존성을 줄이기 위해 사용자가 직접 바꿀 수 있게 둡니다.

- `PROJECT_ROOT`: 레포 루트
- `INPUT_PATH`: 입력 parquet
- `OUTPUT_DIR`: 산출물 저장 위치
- `SAMPLE_ROWS`: sample build 행 수
- `OVERWRITE`: 기존 산출물 덮어쓰기 여부
- `COLUMN_MAP`: 논리 컬럼명 -> 실제 parquet 컬럼명 매핑


In [ ]:
from pathlib import Path
import os
import pandas as pd
import sys
import tempfile

# 노트북 실행 위치가 프로젝트 루트이거나 현재 폴더인 경우를 모두 지원합니다.
CANDIDATE_MODULE_DIRS = (
    Path.cwd(),
    Path.cwd() / 'ml' / 'ml-00_baseline' / 'feature_build',
)
MODULE_DIR = next((path for path in CANDIDATE_MODULE_DIRS if (path / 'fb_utils.py').exists()), None)

if MODULE_DIR is None:
    raise FileNotFoundError(
        'fb_utils.py를 찾지 못했습니다. '
        f'노트북 실행 위치를 확인하세요. candidates={[str(path) for path in CANDIDATE_MODULE_DIRS]}'
    )

if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from fb_utils import BASE_DIR, PROCESSED_DIR

PROJECT_ROOT = BASE_DIR
INPUT_PATH = PROCESSED_DIR / 'step01_clean_base' / 'clean_base.parquet'
TMP_BASE = Path(os.getenv('GRAPH_AML_TMP_DIR', tempfile.gettempdir())).resolve()
TMP_BASE.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix='fb_feature_build_notebook_', dir=TMP_BASE))

EXPERIMENT_ID = 'fb_notebook_sample'
RUN_NAME = 'notebook_user_selected_operations'
SAMPLE_ROWS = 1_000
OVERWRITE = False

# 컬럼명이 바뀌면 이 dict만 고칩니다.
# key는 FeatureSpec에서 쓰는 논리 컬럼명, value는 실제 parquet 컬럼명입니다.
# COLUMN_MAP에 없는 logical key는 fb_schema.py의 COLUMN_CANDIDATES로 fallback됩니다.
# 완전히 명시적으로 관리하려면 필요한 logical key를 COLUMN_MAP에 모두 적으세요.
COLUMN_MAP = {
    'tx_id': 'tx_id',
    'timestamp': 'timestamp',
    'label': 'is_laundering',
    'amount': 'amount',
    'sender_account_id': 'sender_account_id',
    'receiver_account_id': 'receiver_account_id',
    'sender_bank_id': 'sender_bank_id',
    'receiver_bank_id': 'receiver_bank_id',
    'payment_currency': 'Payment Currency',
    'receiving_currency': 'Receiving Currency',
    'payment_format': 'Payment Format',
}

# 이미 split된 raw parquet가 표준 컬럼명을 쓰는 경우 예시입니다.
# 필요하면 COLUMN_MAP = COLUMN_MAP_RAW_SPLIT 로 바꿔서 사용하세요.
COLUMN_MAP_RAW_SPLIT = {
    'tx_id': 'tx_id',
    'timestamp': 'timestamp',
    'label': 'label',
    'amount': 'amount',
    'sender_account_id': 'sender_account',
    'receiver_account_id': 'receiver_account',
    'sender_bank_id': 'from_bank',
    'receiver_bank_id': 'to_bank',
    'payment_currency': 'payment_currency',
    'receiving_currency': 'receiving_currency',
    'payment_format': 'payment_format',
}

# 이미 ML-00 split parquet에서 ML-01 history만 추가할 때 쓰는 최소 매핑입니다.
# 이 경로는 ML-01 전체가 아니라 history-only 산출물입니다.
# 기존 ML-00 feature와 합치려면 별도 join이 필요합니다.
# 이 경우 FEATURE_SPECS_SELECTED는 ML01_TIMEHIST_FEATURE_SPECS만 선택하세요.
COLUMN_MAP_ML00_SPLIT_FOR_ML01_HISTORY = {
    'tx_id': 'tx_id',
    'timestamp': 'timestamp',
    'label': 'label',
    'amount': 'amount__current__value',
    'sender_account_id': 'sender_account',
    'receiver_account_id': 'receiver_account',
}

print('PROJECT_ROOT =', PROJECT_ROOT)
print('INPUT_PATH   =', INPUT_PATH)
print('OUTPUT_DIR  =', OUTPUT_DIR)


## 2. 모듈 import

사용자는 보통 아래 함수만 알면 됩니다.

- `FeatureBuildConfig`, `build_features`: parquet 파일에서 feature build
- `build_features_from_split_paths`: 이미 split된 train/val/test parquet에서 feature build
- `build_features_from_frame`: DataFrame으로 빠른 toy 검증
- `*_spec`: feature 생성 선언 함수


In [2]:
from fb_build import (
    FeatureBuildConfig,
    build_features,
    build_features_from_frame,
    build_features_from_split_paths,
)
from fb_specs import (
    category_code_spec,
    current_value_spec,
    datetime_part_spec,
    default_feature_specs,
    fan_in_spec,
    fan_out_spec,
    is_first_entity_spec,
    log1p_spec,
    recency_seconds_spec,
    rolling_agg_spec,
)


## 3. ML-00 기준 feature set

아래는 기존 `ml_feature_columns.csv`의 ML-00 컬럼명을 기준으로 작성한 예시입니다.

컬럼명이 바뀌면 위의 `COLUMN_MAP`만 고치고, 여기의 `input_col`은 논리 컬럼명으로 유지합니다.

`ML00_FEATURE_SPECS`는 `default_feature_specs()`와 같은 순서입니다. feature order/hash 비교를 위해 순서를 임의로 바꾸지 마세요.

포함 feature: 금액, 은행/통화/결제방식 category code, 현재 row timestamp 파생 변수


In [ ]:
ML00_FEATURE_SPECS = (
    log1p_spec('amount', 'amount__current__log1p'),
    current_value_spec(
        'amount',
        'amount__current__value',
        family='raw_amount',
        aml_typology='large_amount',
    ),
    category_code_spec(
        'sender_bank_id',
        'cat__from_bank__code',
        family='raw_bank',
        aml_typology='cross_institution_flow',
    ),
    category_code_spec(
        'payment_currency',
        'cat__payment_currency__code',
        family='raw_currency',
        aml_typology='currency_conversion',
    ),
    category_code_spec(
        'payment_format',
        'cat__payment_format__code',
        family='raw_payment_format',
        aml_typology='payment_channel',
    ),
    category_code_spec(
        'receiving_currency',
        'cat__receiving_currency__code',
        family='raw_currency',
        aml_typology='currency_conversion',
    ),
    category_code_spec(
        'receiver_bank_id',
        'cat__to_bank__code',
        family='raw_bank',
        aml_typology='cross_institution_flow',
    ),
    datetime_part_spec('timestamp', 'time__row__dayofweek', part='dayofweek'),
    datetime_part_spec('timestamp', 'time__row__hour', part='hour'),
    datetime_part_spec('timestamp', 'time__row__is_weekend', part='is_weekend'),
)

for spec in ML00_FEATURE_SPECS:
    print(f'{spec.output_col:45s} <- {spec.operation:15s} input={dict(spec.input_cols)} params={dict(spec.params)}')


## 4. ML-01 기준 feature set

ML-01은 ML-00에 sender/receiver 과거 window 통계와 recency/first flag를 누적하는 예시입니다.

기존 ML-01 컬럼셋 기준으로 `ML-00 10개 + timehist 30개 + recency/first 4개 = 44개`를 만듭니다.


In [ ]:
ML01_WINDOWS = (
    ('w1h', '1h'),
    ('w6h', '6h'),
    ('w1d', '1d'),
    ('w3d', '3d'),
    ('w7d', '7d'),
)

def make_ml01_timehist_specs(entity_name, direction, entity_col):
    specs = []
    for suffix, window in ML01_WINDOWS:
        prefix = f'timehist__{entity_name}__{direction}'
        specs.extend([
            rolling_agg_spec(
                entity_col,
                'timestamp',
                'amount',
                f'{prefix}__tx_count__count__{suffix}',
                window=window,
                agg='count',
                family='timehist',
                aml_typology='velocity',
            ),
            rolling_agg_spec(
                entity_col,
                'timestamp',
                'amount',
                f'{prefix}__amount__sum__{suffix}',
                window=window,
                agg='sum',
                family='timehist',
                aml_typology='amount_burst',
            ),
            rolling_agg_spec(
                entity_col,
                'timestamp',
                'amount',
                f'{prefix}__amount__mean__{suffix}',
                window=window,
                agg='mean',
                family='timehist',
                aml_typology='average_behavior',
            ),
        ])
    return tuple(specs)

ML01_SENDER_TIMEHIST_FEATURE_SPECS = make_ml01_timehist_specs('sender', 'out', 'sender_account_id')
ML01_RECEIVER_TIMEHIST_FEATURE_SPECS = make_ml01_timehist_specs('receiver', 'in', 'receiver_account_id')
ML01_TIMEHIST_FEATURE_SPECS = (
    *ML01_SENDER_TIMEHIST_FEATURE_SPECS,
    *ML01_RECEIVER_TIMEHIST_FEATURE_SPECS,
)

ML01_SENDER_RECENCY_FEATURE_SPECS = (
    recency_seconds_spec(
        'sender_account_id',
        'timestamp',
        'recency__sender__out__seconds_since_last',
        aml_typology='recency',
    ),
    is_first_entity_spec(
        'sender_account_id',
        'timestamp',
        'flag__sender__out__is_first_tx',
        aml_typology='cold_start',
    ),
)

ML01_RECEIVER_RECENCY_FEATURE_SPECS = (
    recency_seconds_spec(
        'receiver_account_id',
        'timestamp',
        'recency__receiver__in__seconds_since_last',
        aml_typology='recency',
    ),
    is_first_entity_spec(
        'receiver_account_id',
        'timestamp',
        'flag__receiver__in__is_first_tx',
        aml_typology='cold_start',
    ),
)

ML01_ADDED_FEATURE_SPECS = (
    *ML01_SENDER_TIMEHIST_FEATURE_SPECS,
    *ML01_SENDER_RECENCY_FEATURE_SPECS,
    *ML01_RECEIVER_TIMEHIST_FEATURE_SPECS,
    *ML01_RECEIVER_RECENCY_FEATURE_SPECS,
)
ML01_FEATURE_SPECS = (*ML00_FEATURE_SPECS, *ML01_ADDED_FEATURE_SPECS)

# 기본 실행은 가벼운 ML-00입니다. ML-01을 실행하려면 아래 두 줄만 바꾸세요.
FEATURE_SET_NAME = 'ML-00'
FEATURE_SPECS_SELECTED = ML00_FEATURE_SPECS
# FEATURE_SET_NAME = 'ML-01'
# FEATURE_SPECS_SELECTED = ML01_FEATURE_SPECS
# 이미 ML-00 split parquet에서 history만 새로 만들 때:
# COLUMN_MAP = COLUMN_MAP_ML00_SPLIT_FOR_ML01_HISTORY
# FEATURE_SET_NAME = 'ML-01-history-only'
# FEATURE_SPECS_SELECTED = ML01_TIMEHIST_FEATURE_SPECS

print('selected:', FEATURE_SET_NAME, 'feature_count:', len(FEATURE_SPECS_SELECTED))
print('ML-00 feature_count:', len(ML00_FEATURE_SPECS))
print('ML-01 feature_count:', len(ML01_FEATURE_SPECS))
for spec in ML01_TIMEHIST_FEATURE_SPECS[:6]:
    print(f'{spec.output_col:55s} <- {spec.operation:12s} input={dict(spec.input_cols)} params={dict(spec.params)}')


## 5. 예시 C: GFP 스타일 graph/temporal operation

아래 feature들은 GFP 선행연구의 primitive 아이디어를 단순화한 예시입니다.

주의:

- `sender_account_id`, `receiver_account_id` 컬럼이 입력 데이터에 있어야 합니다.
- 현재 `clean_base.parquet`에 해당 컬럼명이 없으면 실행 시 명시적 에러가 납니다.
- rolling/fan feature는 `closed='left'` 정책으로 현재 row와 미래 row를 제외합니다.


In [ ]:
FEATURE_SPECS_GRAPH_EXAMPLE = (
    rolling_agg_spec(
        entity_col='sender_account_id',
        timestamp_col='timestamp',
        value_col='amount',
        output_col='sender__amount_sum__7d',
        window='7d',
        agg='sum',
    ),
    fan_out_spec(
        sender_col='sender_account_id',
        receiver_col='receiver_account_id',
        timestamp_col='timestamp',
        output_col='sender__fan_out_unique_receiver__7d',
        window='7d',
    ),
    fan_in_spec(
        receiver_col='receiver_account_id',
        sender_col='sender_account_id',
        timestamp_col='timestamp',
        output_col='receiver__fan_in_unique_sender__7d',
        window='7d',
    ),
)

for spec in FEATURE_SPECS_GRAPH_EXAMPLE:
    print(f'{spec.output_col:45s} <- {spec.operation:12s} input={dict(spec.input_cols)} params={dict(spec.params)}')


## 6. sample parquet build 실행

기본값은 `SAMPLE_ROWS=1_000`입니다.

처음에는 full data가 아니라 sample로 아래 파일들을 확인하세요.

이미 train/val/test parquet가 나뉘어 있으면 `build_features_from_split_paths()`를 사용합니다. 단, feature 계산에 필요한 원천 컬럼이 각 split 파일에 있어야 합니다.

- `{experiment_id}_Xy_train.parquet`
- `{experiment_id}_Xy_val.parquet`
- `{experiment_id}_Xy_test.parquet`
- `ml_feature_columns.csv`
- `feature_catalog.csv`
- `feature_info.csv`
- `category_mapping_train_only.csv`
- `category_unknown_summary.csv`


In [ ]:
config = FeatureBuildConfig(
    base_dir=PROJECT_ROOT,
    input_path=INPUT_PATH,
    output_dir=OUTPUT_DIR,
    experiment_id=EXPERIMENT_ID,
    run_name=RUN_NAME,
    feature_specs=FEATURE_SPECS_SELECTED,
    column_map=COLUMN_MAP,
    sample_rows=SAMPLE_ROWS,
    overwrite=OVERWRITE,
)

result = build_features(config)

print('row_counts:', result.row_counts)
print('feature_columns:', result.feature_columns)
print('unknown_category_total:', result.build_summary['unknown_category_total'])
print('output_dir:', result.output_paths.output_dir)

# 이미 split된 parquet 3개를 입력으로 쓸 때 예시입니다. 필요할 때 경로만 채워서 사용하세요.
# result = build_features_from_split_paths(
#     train_path=TRAIN_PATH,
#     val_path=VAL_PATH,
#     test_path=TEST_PATH,
#     output_dir=OUTPUT_DIR,
#     experiment_id=EXPERIMENT_ID,
#     run_name=RUN_NAME,
#     feature_specs=FEATURE_SPECS_SELECTED,
#     column_map=COLUMN_MAP,
#     sample_rows=SAMPLE_ROWS,
#     overwrite=OVERWRITE,
# )


## 7. 생성 feature와 feature info 확인

`feature_frame`은 메모리에도 반환됩니다.

`feature_info`는 operation이 생성한 컬럼별 분포/품질 정보입니다.


In [ ]:
display(result.feature_frame.head())
display(result.feature_info)


## 8. 산출물 CSV 확인

모델 입력 truth source는 `ml_feature_columns.csv`입니다.

`feature_catalog.csv`는 사람이 검토하는 설명/관리용 파일입니다.


In [ ]:
display(pd.read_csv(result.output_paths.feature_columns_path))
display(pd.read_csv(result.output_paths.feature_catalog_path))
display(pd.read_csv(result.output_paths.category_unknown_summary_path))


## 9. DataFrame toy example

파일 없이 작은 DataFrame으로 operation 동작을 빠르게 확인할 수 있습니다.

이 예시는 rolling/fan-in/fan-out이 현재 row 이전 데이터만 보는지 확인하는 용도입니다.


In [ ]:
toy_df = pd.DataFrame({
    'tx_id': ['t1', 't2', 't3', 't4', 't5', 't6'],
    'timestamp': pd.to_datetime([
        '2022-01-01 00:00:00',
        '2022-01-01 01:00:00',
        '2022-01-02 00:00:00',
        '2022-01-03 00:00:00',
        '2022-01-04 00:00:00',
        '2022-01-05 00:00:00',
    ]),
    'label': [0, 0, 1, 0, 0, 1],
    'sender_account_id': ['a', 'a', 'b', 'a', 'c', 'a'],
    'receiver_account_id': ['x', 'y', 'a', 'z', 'a', 'x'],
    'amount': [10, 20, 30, 40, 50, 60],
})

toy_specs = (
    current_value_spec('amount', 'amount__value'),
    rolling_agg_spec('sender_account_id', 'timestamp', 'amount', 'sender__amount_sum__2d', window='2d', agg='sum'),
    fan_out_spec('sender_account_id', 'receiver_account_id', 'timestamp', 'sender__fan_out__2d', window='2d'),
    fan_in_spec('receiver_account_id', 'sender_account_id', 'timestamp', 'receiver__fan_in__2d', window='2d'),
)

toy_result = build_features_from_frame(
    toy_df,
    feature_specs=toy_specs,
    output_dir=None,
)

display(toy_result.feature_frame)
display(toy_result.feature_info)


## 10. full data 실행 전 체크리스트

full data feature generation은 시간이 걸릴 수 있으므로 승인 후 실행하세요.

실행 전 확인:

- sample build가 성공했는가
- `feature_catalog.csv`의 입력 컬럼/operation/params가 의도와 맞는가
- `feature_info.csv`에 missing/inf 문제가 없는가
- `category_unknown_summary.csv`의 unknown category 규모가 허용 가능한가
- `ml_feature_columns.csv`의 `used_in_ml`이 의도와 맞는가
- 기존 산출물을 덮어쓸 것인지 명시적으로 결정했는가


In [ ]:
# full data 실행 예시입니다. 기본적으로 실행하지 마세요.
RUN_FULL_BUILD = False

if RUN_FULL_BUILD:
    full_config = FeatureBuildConfig(
        base_dir=PROJECT_ROOT,
        input_path=INPUT_PATH,
        output_dir=PROJECT_ROOT / 'ml' / 'ml-00_baseline' / 'outputs' / 'feature_build' / 'fb_full_user_selected' / 'run_001',
        experiment_id='fb_full_user_selected',
        run_name='full_user_selected_operations',
        feature_specs=FEATURE_SPECS_SELECTED,
        column_map=COLUMN_MAP,
        sample_rows=None,
        overwrite=False,
    )
    full_result = build_features(full_config)
    print(full_result.row_counts)
else:
    print('RUN_FULL_BUILD=False: full data build skipped.')
